Kelly Quinn 
September 2026
"Does Outsourcing Still Cost Workers? Revisiting the Contractor Wage Penalty for Janitors and Security Guards, 2001–2025"

In [1]:
import numpy as np
import pandas as pd
from ipumspy import readers

ddi = readers.read_ipums_ddi("../data/ipums_data/cps_00047.xml")  # codebook only, fast
dk_raw  = pd.read_parquet("../data/ipums_data/dk_core.parquet")
dk_core = dk_raw[
    dk_raw["OCC1990"].isin([453, 426])  # explicit filter, even though dk_core.parquet is already restricted
    & (dk_raw["AGE"] >= 16)
    & (dk_raw["YEAR"] >= 1983) & (dk_raw["YEAR"] <= 2025) & (dk_raw["YEAR"] != 2020)
].copy()
print(dk_raw.shape, dk_core.shape)

/Users/kq/opt/anaconda3/envs/outsourcing/lib/python3.12/site-packages/ipumspy/readers.py:70: CitationWarning: Use of data from IPUMS is subject to conditions including that users should cite the data appropriately.
See the `ipums_conditions` attribute of this codebook for terms of use.
See the `ipums_citation` attribute of this codebook for the appropriate citation.
  warnings.warn(


(171905, 22) (166391, 22)


In [2]:
# check cells and compare to r df 
print(dk_core["YEAR"].min(), dk_core["YEAR"].max())
print(dk_core["OCC1990"].value_counts())
print((dk_core["EARNWT"] > 0).sum())
print(dk_core.loc[dk_core["EARNWT"] > 0, "MISH"].value_counts().sort_index())

1983 2025
OCC1990
453    122524
426     43867
Name: count, dtype: Int64
166391
MISH
4    82990
8    83401
Name: count, dtype: Int64


In [3]:
# codebook check before recoding 
for var in ["UHRSWORKORG", "UHRSWORK1", "HOURWAGE", "EARNWEEK2", "PAIDHOUR", "UNION", "METRO", "CLASSWKR"]:
    print(var, ddi.get_variable_info(var).codes, "\n")

UHRSWORKORG {} 

UHRSWORK1 {'0 hours': 0, 'Hours vary': 997, 'NIU/Missing': 999} 

HOURWAGE {} 

EARNWEEK2 {} 

PAIDHOUR {'NIU': 0, 'No': 1, 'Yes': 2, 'Refused': 6, "Don't Know": 7} 

UNION {'NIU': 0, 'No union coverage': 1, 'Member of labor union': 2, 'Covered by union but not a member': 3} 

METRO {'Not identified': 0, 'Not in metropolitan area': 1, 'In central/principal city': 2, 'Not in central/principal city': 3, 'Central/principal city status not identified': 4, 'Missing/unknown': 9} 

CLASSWKR {'NIU': 0, 'Self-employed': 10, 'Self-employed, not incorporated': 13, 'Self-employed, incorporated': 14, 'Works for wages or salary': 20, 'Wage/salary, private': 21, 'Private, for profit': 22, 'Private, nonprofit': 23, 'Wage/salary, government': 24, 'Federal government employee': 25, 'Armed forces': 26, 'State government employee': 27, 'Local government employee': 28, 'Unpaid family worker': 29, 'Missing/Unknown': 99} 



In [4]:
for var in ["UHRSWORKORG", "HOURWAGE", "HOURWAGE2", "EARNWEEK2"]:
    s = dk_core[var]
    print(var, "max:", s.max())
    print(s.value_counts().sort_index().tail(5), "\n")

UHRSWORKORG max: 999
UHRSWORKORG
96         2
98         3
99        13
998        4
999    43254
Name: count, dtype: Int64 

HOURWAGE max: 999.99
HOURWAGE
98.00         1
98.85         1
99.00         2
99.50         1
999.99    29878
Name: count, dtype: int64 

HOURWAGE2 max: 999.99
HOURWAGE2
71.35         1
72.20         1
72.41         1
76.30         2
999.99    31060
Name: count, dtype: int64 

EARNWEEK2 max: 999999.99
EARNWEEK2
9356.71      1
11256.27     1
11952.53     1
13258.23     2
999999.99    1
Name: count, dtype: int64 



## Build dk_clean: recode variables per refs/cleaning.md

dk_core -> dk_clean. Constructs the outsourcing indicator, hourly/real wage,
and all controls in refs/cleaning.md (union, part-time, age/age2, sex, race,
education categories, four-way urbanicity, gov_employee, interactions), plus
the cluster variable (survey month) and sample-period flags. No rows are
dropped here — different specifications use different subsets of covariates,
so listwise deletion is left to the estimation step.

In [5]:
# codebook check before recoding demographics/education
for var in ["RACE", "HISPAN", "EDUC", "SEX"]:
    print(var, ddi.get_variable_info(var).codes, "\n")

RACE {'White': 100, 'Black': 200, 'American Indian/Aleut/Eskimo': 300, 'Asian or Pacific Islander': 650, 'Asian only': 651, 'Hawaiian/Pacific Islander only': 652, 'Other (single) race, n.e.c.': 700, 'White-Black': 801, 'White-American Indian': 802, 'White-Asian': 803, 'White-Hawaiian/Pacific Islander': 804, 'Black-American Indian': 805, 'Black-Asian': 806, 'Black-Hawaiian/Pacific Islander': 807, 'American Indian-Asian': 808, 'Asian-Hawaiian/Pacific Islander': 809, 'White-Black-American Indian': 810, 'White-Black-Asian': 811, 'White-American Indian-Asian': 812, 'White-Asian-Hawaiian/Pacific Islander': 813, 'White-Black-American Indian-Asian': 814, 'American Indian-Hawaiian/Pacific Islander': 815, 'White-Black--Hawaiian/Pacific Islander': 816, 'White-American Indian-Hawaiian/Pacific Islander': 817, 'Black-American Indian-Asian': 818, 'White-American Indian-Asian-Hawaiian/Pacific Islander': 819, 'Two or three races, unspecified': 820, 'Four or five races, unspecified': 830, 'Blank': 999} 

In [6]:
dk_clean = dk_core.copy()

# missing-value sentinels -> NaN (see refs/cleaning.md). 0 is missing for wages
# and hours: a wage/salary worker reporting current earnings cannot have a true
# wage of 0, and 0 usual hours carries no information about hours worked.
sentinels = {
    "HOURWAGE2":   [0, 999.99],
    "EARNWEEK2":   [0, 999999.99],
    "UHRSWORKORG": [0, 998, 999],
    "UHRSWORK1":   [0, 997, 999],
}
for col, codes in sentinels.items():
    dk_clean[col] = dk_clean[col].replace(codes, np.nan)

# hours: UHRSWORKORG (matched to ORG earnings); fall back to UHRSWORK1 when missing.
# Used for both the wage denominator and parttime.
dk_clean["hours"] = dk_clean["UHRSWORKORG"].fillna(dk_clean["UHRSWORK1"])

# hourly wage: HOURWAGE2 for paid-hourly workers with a valid rate, else weekly
# earnings / hours (also covers paid-hourly workers with missing HOURWAGE2).
# HOURWAGE2 for all years: HOURWAGE is unpopulated from April 2023; HOURWAGE2 applies
# post-2023 rounding/topcoding to all years. .isin()/.notna() return plain bool arrays
# on nullable Int64; fillna(0) before > avoids masked arrays in np.select.
paidhourly_valid = dk_clean["PAIDHOUR"].isin([2]) & dk_clean["HOURWAGE2"].notna()
weekly_valid = dk_clean["EARNWEEK2"].notna() & (dk_clean["hours"].fillna(0).to_numpy(dtype=float) > 0)
dk_clean["hourly_wage"] = np.select(
    [paidhourly_valid, weekly_valid],
    [dk_clean["HOURWAGE2"], dk_clean["EARNWEEK2"] / dk_clean["hours"]],
    default=np.nan,
)
dk_clean["log_wage_untrimmed"] = np.log(dk_clean["hourly_wage"].where(dk_clean["hourly_wage"] > 0))

# real_wage: 2025 dollars, annual-average CPI-U-RS; summary stats only
cpi = pd.read_csv("../data/cpi_annual.csv").set_index("YEAR")["AVG"]
dk_clean["real_wage"] = dk_clean["hourly_wage"] * (cpi.loc[2025] / dk_clean["YEAR"].map(cpi))

# Trim implausible wages: below half the federal minimum in that year, or above
# $100/hr in 2025 dollars. Janitors and guards are FLSA-covered, so values outside
# this range are treated as measurement error. Rows are kept (incidence doesn't
# use wages); only the wage variables are set to NaN.
mw = pd.read_csv("../data/minimumwage_annual.csv")
fed = mw.assign(YEAR=mw["observation_date"].str[:4].astype(int)).set_index("YEAR")["STTMINWGFG"]

dk_clean["wage_trimmed"] = (
    (dk_clean["hourly_wage"] < 0.5 * dk_clean["YEAR"].map(fed))
    | (dk_clean["real_wage"] > 100)
)
dk_clean["log_wage"] = dk_clean["log_wage_untrimmed"].where(~dk_clean["wage_trimmed"])
dk_clean["real_wage_trimmed"] = dk_clean["real_wage"].where(~dk_clean["wage_trimmed"])

# checks
print(dk_clean[["hours", "hourly_wage", "log_wage", "real_wage_trimmed"]].describe())
print("zero wages:", (dk_clean["hourly_wage"] == 0).sum())
print("real_wage NaN beyond hourly_wage NaN:",
      dk_clean["real_wage"].isna().sum() - dk_clean["hourly_wage"].isna().sum())
print("trimmed:", dk_clean["wage_trimmed"].sum())
print(dk_clean.groupby("OCC1990")["wage_trimmed"].mean().round(4))

          hours    hourly_wage       log_wage  real_wage_trimmed
count  163977.0  165016.000000  163878.000000      163878.000000
mean   34.57519      10.784249       2.219862          18.503072
std    11.22259       8.530271       0.543677           9.247253
min         1.0       0.050000       0.526093           3.800000
25%        30.0       6.150000       1.832581          12.499844
50%        40.0       9.000000       2.197225          16.063378
75%        40.0      13.000000       2.564949          21.607009
max        99.0     750.000000       4.564348          98.771440
zero wages: 0
real_wage NaN beyond hourly_wage NaN: 0
trimmed: 1138
OCC1990
426    0.0040
453    0.0079
Name: wage_trimmed, dtype: float64


In [7]:
# outsourcing status: janitors (453) in Services to Buildings & Dwellings (722),
# guards (426) in Protective Services (740)
dk_clean["outsourced"] = (
    ((dk_clean["OCC1990"] == 453) & (dk_clean["IND1990"] == 722))
    | ((dk_clean["OCC1990"] == 426) & (dk_clean["IND1990"] == 740))
).astype(int)
dk_clean["occupation"] = np.where(dk_clean["OCC1990"] == 453, "janitor", "guard")

dk_clean["union"] = dk_clean["UNION"].map({1: 0, 2: 1, 3: 1})  # member or covered; NIU (0) -> NaN

print(dk_clean.groupby("occupation")["outsourced"].mean())
print("union NaN:", dk_clean["union"].isna().sum())

occupation
guard      0.436456
janitor    0.184903
Name: outsourced, dtype: float64
union NaN: 1698


In [ ]:
# parttime: from the combined "hours" variable (same as the wage denominator:
# UHRSWORKORG, falling back to UHRSWORK1). Threshold follows the BLS definition of
# part-time work (< 35 usual hours). 0 hours is already NaN via the sentinels.
hrs = dk_clean["hours"].to_numpy(dtype="float64", na_value=np.nan)
dk_clean["parttime"] = np.where(np.isnan(hrs), np.nan, (hrs < 35).astype(float))

print("parttime NaN:", dk_clean["parttime"].isna().sum())
print(dk_clean["parttime"].value_counts(dropna=False))

UHRSWORK1 == 0 count: 0
parttime NaN: 6347


Series([], Name: count, dtype: int64)

In [9]:
dk_clean["age"] = dk_clean["AGE"]
dk_clean["age2"] = dk_clean["age"] ** 2
dk_clean["female"] = (dk_clean["SEX"] == 2).astype(int)

# hispanic: 1 if HISPAN 100-612, 0 if HISPAN == 0, NaN if 901 (don't know) / 902 (N/A).
# Cast to a plain numpy array first -- comparisons on nullable Int64 columns return
# a masked BooleanArray, which np.select rejects.
hispan_arr = dk_clean["HISPAN"].to_numpy(dtype="float64", na_value=np.nan)
dk_clean["hispanic"] = np.select(
    [hispan_arr == 0, (hispan_arr >= 100) & (hispan_arr <= 612), np.isin(hispan_arr, [901, 902])],
    [0, 1, np.nan],
    default=np.nan,
)

# black: 1 if RACE==200 (single-race Black) and not Hispanic; 0 if RACE!=200 or
# Hispanic; NaN if hispanic is NaN. Race/ethnicity are coded as mutually exclusive.
race_arr = dk_clean["RACE"].to_numpy(dtype="float64", na_value=np.nan)
dk_clean["black"] = np.select(
    [dk_clean["hispanic"].isna(), (race_arr == 200) & (dk_clean["hispanic"] == 0)],
    [np.nan, 1],
    default=0,
)

# D&K's six education categories; college is the omitted/reference category, so it's
# listed first in `categories` (determines the reference level downstream)
educ_bins = {
    "no_school": [2],
    "primary": [10, 11, 12, 13, 14, 20, 21, 22, 30, 31, 32],
    "hs_attend": [40, 50, 60, 70, 71, 72],
    "hs_complete": [73],
    "some_college": [80, 81, 90, 91, 92, 100],
    "college": [110, 111, 120, 121, 122, 123, 124, 125],
}
dk_clean["educ_cat"] = pd.NA
for label, codes in educ_bins.items():
    dk_clean.loc[dk_clean["EDUC"].isin(codes), "educ_cat"] = label
educ_categories = ["college"] + [c for c in educ_bins if c != "college"]
dk_clean["educ_cat"] = pd.Categorical(dk_clean["educ_cat"], categories=educ_categories, ordered=False)

print("hispanic NaN:", dk_clean["hispanic"].isna().sum())
print("black NaN:", dk_clean["black"].isna().sum())
print(dk_clean["educ_cat"].value_counts(dropna=False))
print("educ_cat categories (first = reference):", dk_clean["educ_cat"].cat.categories.tolist())

hispanic NaN: 1035
black NaN: 1035
educ_cat
hs_complete     56335
hs_attend       45299
some_college    36290
primary         15994
college         11477
no_school         996
Name: count, dtype: int64
educ_cat categories (first = reference): ['college', 'no_school', 'primary', 'hs_attend', 'hs_complete', 'some_college']


In [10]:
# Urbanicity: four 0/1 dummies, never NaN; reference = nonmetro (METRO == 1).
metro_arr = dk_clean["METRO"].to_numpy(dtype="float64", na_value=np.nan)
dk_clean["msa"] = np.isin(metro_arr, [2, 3, 4]).astype(int)
dk_clean["central_city"] = (metro_arr == 2).astype(int)
dk_clean["cc_unknown"] = (metro_arr == 4).astype(int)  # in MSA, central city status unknown
dk_clean["geo_unidentified"] = np.isin(metro_arr, [0, 9]).astype(int)

dk_clean["ym"] = dk_clean["YEAR"] * 100 + dk_clean["MONTH"]  # cluster variable (survey month)
dk_clean["replication_sample"] = dk_clean["YEAR"].between(1983, 2000)
dk_clean["extension_sample"] = dk_clean["YEAR"] >= 2001  # 2020 already excluded in dk_core

print(dk_clean[["msa", "central_city", "cc_unknown", "geo_unidentified"]].isna().sum())
print(dk_clean[["msa", "central_city", "cc_unknown", "geo_unidentified"]].mean())

msa                 0
central_city        0
cc_unknown          0
geo_unidentified    0
dtype: int64
msa                 0.695512
central_city        0.289276
cc_unknown          0.096628
geo_unidentified    0.074355
dtype: float64


In [11]:
# Government employee flag (kq spec only; not in D&K)
# 1 = government wage/salary; 0 = private wage/salary; NaN = self-employed,
# unpaid, armed forces, generic "works for wages" (20), NIU, missing
dk_clean["gov_employee"] = np.select(
    [dk_clean["CLASSWKR"].isin([24, 25, 27, 28]),
     dk_clean["CLASSWKR"].isin([21, 22, 23])],
    [1, 0],
    default=np.nan,
)

# Interactions (D&K row 4 only)
dk_clean["union_x_outsourced"] = dk_clean["union"] * dk_clean["outsourced"]
dk_clean["parttime_x_outsourced"] = dk_clean["parttime"] * dk_clean["outsourced"]

In [12]:
constructed_vars = [
    "hours", "hourly_wage", "log_wage_untrimmed", "log_wage", "wage_trimmed",
    "real_wage", "real_wage_trimmed", "outsourced", "occupation", "union",
    "parttime", "age", "age2", "female", "hispanic", "black", "educ_cat",
    "msa", "central_city", "cc_unknown", "geo_unidentified", "gov_employee",
    "union_x_outsourced", "parttime_x_outsourced", "ym", "replication_sample",
    "extension_sample",
]
print("dk_clean shape:", dk_clean.shape)
print(dk_clean[constructed_vars].isna().sum())

dk_clean shape: (166391, 49)
hours                    2414
hourly_wage              1375
log_wage_untrimmed       1375
log_wage                 2513
wage_trimmed                0
real_wage                1375
real_wage_trimmed        2513
outsourced                  0
occupation                  0
union                    1698
parttime                 6347
age                         0
age2                        0
female                      0
hispanic                 1035
black                    1035
educ_cat                    0
msa                         0
central_city                0
cc_unknown                  0
geo_unidentified            0
gov_employee                0
union_x_outsourced       1698
parttime_x_outsourced    6347
ym                          0
replication_sample          0
extension_sample            0
dtype: int64


In [13]:
# checks before saving
print(dk_clean.shape)
print(dk_clean[["hourly_wage", "log_wage", "parttime", "union",
                "union_x_outsourced", "parttime_x_outsourced"]].isna().sum())

from pathlib import Path
out_path = Path("../outputs/py/dk_clean_v2.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)
if out_path.exists():
    raise FileExistsError(f"{out_path} already exists; stop and ask")
dk_clean.to_parquet(out_path, index=False)
print("wrote", out_path)

(166391, 49)
hourly_wage              1375
log_wage                 2513
parttime                 6347
union                    1698
union_x_outsourced       1698
parttime_x_outsourced    6347
dtype: int64
wrote ../outputs/py/dk_clean_v2.parquet
